# **Environment Setup**

In [11]:
%pwd

'/home/mmuuser/Desktop/ry/BLIP-VLU-Net/VLU-Net'

In [ ]:
# %cd VLU-Net

# import conda
# import pip
 
# conda create -n vlunet

# conda activate vlunet

# pip install -r requirements.txt

[Errno 2] No such file or directory: 'VLU-Net'
/home/mmuuser/Desktop/ry/BLIP-VLU-Net/VLU-Net


# **Pretrained Checkpoint**

## **VLU-Net weights**
Download the weights from link: https://drive.google.com/file/d/1214SfTO5LDMr3Ck_aVVUuZhuqhteMJ7P/view?usp=sharing, and put it as ./pretrained_ckpt.

In [14]:
from pathlib import Path

ckpt_dir = Path("pretrained_ckpt")
ckpt_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    "single_rain_vlunet.ckpt",
    "single_haze_vlunet.ckpt",
    "single_noise_vlunet.ckpt",
    "single_blur_vlunet.ckpt",
    "single_lowlight_vlunet.ckpt",
    "3task_vlunet.ckpt",
    "5task_vlunet.ckpt",
    "clip_tuned.pth"
]

# Google Drive file ID
gdrive_url = "https://drive.google.com/uc?id=1214SfTO5LDMr3Ck_aVVUuZhuqhteMJ7P"

missing = []

if not ckpt_dir.exists():
    missing = required_files
else:
    for file in required_files:
        if not (ckpt_dir / file).exists():
            missing.append(file)

if not missing:
    print("✓ All pretrained checkpoints found.")
    print(f"Location: {ckpt_dir.resolve()}")
else:
    print("❌ Missing pretrained checkpoints:")
    for file in missing:
        print(f"   - {file}")

    print("\nPlease download the pretrained checkpoint package from:")
    print("https://drive.google.com/file/d/1214SfTO5LDMr3Ck_aVVUuZhuqhteMJ7P/view")
    print(f"\nAfter downloading, extract the contents into the '{ckpt_dir}' directory.")


✓ All pretrained checkpoints found.
Location: /home/mmuuser/Desktop/ry/BLIP-VLU-Net/VLU-Net/pretrained_ckpt


# **Dataset Setup**

In [15]:
%pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [kagglehub]/3 [kagglesdk]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import kagglehub
import shutil
import os
import torch
import numpy as np
from PIL import Image

/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Deblurring**

In [10]:
blur_target_dir = os.path.join("datasets", "deblurring_datasets", "GoPro")
os.makedirs(blur_target_dir, exist_ok=True)
image_count = 0

if os.path.exists(blur_target_dir):
    for root, dirs, files in os.walk(blur_target_dir):
        image_count += len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

if image_count > 0:
    print(f"✓ Dataset already exists")
    print(f"✓ Found {image_count} images")
    print("Skipping download")
else:
    print("Dataset not found. Downloading...")
    os.makedirs(os.path.dirname(blur_target_dir), exist_ok=True)
    cache_path = kagglehub.dataset_download("adwythdarsanr/gopro-image-deblurring-dataset")
    print(f"Downloaded to cache: {cache_path}")
    gopro_source = os.path.join(cache_path, "Gopro")
    if not os.path.exists(gopro_source):
        gopro_source = cache_path
    os.makedirs(blur_target_dir, exist_ok=True)
    for item in os.listdir(gopro_source):
        source = os.path.join(gopro_source, item)
        destination = os.path.join(blur_target_dir, item)
        if os.path.isdir(source):
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(source, destination)
    print(f"✓ Dataset copied to {blur_target_dir}")

✓ Dataset already exists
✓ Found 6428 images
Skipping download


## **Dehazing**

### **Training Dataset**

In [2]:
haze_base_target_dir = os.path.join("datasets", "dehazing_datasets")
os.makedirs(haze_base_target_dir, exist_ok=True)

# Training Dataset (OTS)
train_target_dir = os.path.join(haze_base_target_dir, "OTS_outdoors")

train_count = 0
if os.path.exists(train_target_dir):
    for root, _, files in os.walk(train_target_dir):
        train_count += len(files)

if train_count > 0:
    print(f"✓ OTS Training already exists ({train_count} files). Skipping download.")
else:
    print("Downloading OTS Training dataset...")

    cache_path = kagglehub.dataset_download(
        "brunobelloni/outdoor-training-set-ots-reside"
    )

    print(f"Cache: {cache_path}")

    os.makedirs(train_target_dir, exist_ok=True)

    for item in os.listdir(cache_path):
        source = os.path.join(cache_path, item)
        destination = os.path.join(train_target_dir, item)

        if os.path.isdir(source):
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(source, destination)

    print(f"✓ OTS Training ready at: {train_target_dir}")

Resuming download from 4392484864 bytes (7485435986 bytes left)...
Resuming download to /home/mmuuser/.cache/kagglehub/datasets/brunobelloni/outdoor-training-set-ots-reside/3.archive (4392484864/11877920850) bytes left.


100%|██████████| 11.1G/11.1G [25:37<00:00, 4.87MB/s]  

Extracting files...


Cache: /home/mmuuser/.cache/kagglehub/datasets/brunobelloni/outdoor-training-set-ots-reside/versions/3
✓ OTS Training ready at: datasets/dehazing_datasets/OTS_outdoors


### **Testing Dataset**

In [3]:
# Testing Dataset (SOTS)
test_target_dir = os.path.join(haze_base_target_dir, "SOTS_outdoors")

test_count = 0
if os.path.exists(test_target_dir):
    for root, _, files in os.walk(test_target_dir):
        test_count += len(files)

if test_count > 0:
    print(f"✓ SOTS Testing already exists ({test_count} files). Skipping download.")
else:
    print("Downloading SOTS Testing dataset...")

    cache_path = kagglehub.dataset_download(
        "balraj98/synthetic-objective-testing-set-sots-reside"
    )

    print(f"Cache: {cache_path}")

    source_outdoor_path = os.path.join(cache_path, "outdoor")

    folders_to_move = ["clear", "hazy"]

    os.makedirs(test_target_dir, exist_ok=True)

    for folder in folders_to_move:
        source_folder = os.path.join(source_outdoor_path, folder)
        destination_folder = os.path.join(test_target_dir, folder)

        if os.path.exists(source_folder):

            if os.path.exists(destination_folder):
                shutil.rmtree(destination_folder)

            shutil.move(source_folder, destination_folder)

            print(f"Moved {folder} → {destination_folder}")

        else:
            print(f"Warning: {source_folder} not found")

    print(f"✓ SOTS Testing ready at: {test_target_dir}")

100%|██████████| 415M/415M [00:39<00:00, 11.0MB/s] 

Extracting files...


Cache: /home/mmuuser/.cache/kagglehub/datasets/balraj98/synthetic-objective-testing-set-sots-reside/versions/1
Moved clear → datasets/dehazing_datasets/SOTS_outdoors/clear
Moved hazy → datasets/dehazing_datasets/SOTS_outdoors/hazy
✓ SOTS Testing ready at: datasets/dehazing_datasets/SOTS_outdoors


## **Delowlight**

In [4]:
lowlight_base_target_dir = os.path.join("datasets", "delowlight_datasets", "LoL")
os.makedirs(lowlight_base_target_dir, exist_ok=True)

image_count = 0

if os.path.exists(lowlight_base_target_dir):
    for root, _, files in os.walk(lowlight_base_target_dir):
        image_count += len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

if image_count > 0:
    print(f"✓ LoL dataset already exists ({image_count} images). Skipping download.")

else:
    print("Downloading LOL dataset...")
    cache_path = kagglehub.dataset_download("soumikrakshit/lol-dataset")

    print(f"Cache path: {cache_path}")
    source_root = os.path.join(cache_path, "lol_dataset")

    if not os.path.exists(source_root):
        source_root = cache_path

    folders_to_copy = ["eval15", "our485"]
    print(f"Copying dataset into: {lowlight_base_target_dir}")
    
    for folder in folders_to_copy:
        source_folder_path = os.path.join(source_root, folder)
        destination_folder_path = os.path.join(lowlight_base_target_dir, folder)

        if os.path.exists(source_folder_path):

            if os.path.exists(destination_folder_path): shutil.rmtree(destination_folder_path)
            shutil.copytree(source_folder_path, destination_folder_path)

            print(f"✓ Copied: {folder}")

        else:
            print(f"⚠ Missing folder: {source_folder_path}")

    print("\n✓ LoL dataset ready at:")
    print(f"- {lowlight_base_target_dir}/eval15")
    print(f"- {lowlight_base_target_dir}/our485")

100%|██████████| 331M/331M [00:31<00:00, 10.9MB/s] 

Extracting files...


Cache path: /home/mmuuser/.cache/kagglehub/datasets/soumikrakshit/lol-dataset/versions/1
Copying dataset into: datasets/delowlight_datasets/LoL
✓ Copied: eval15
✓ Copied: our485

✓ LoL dataset ready at:
- datasets/delowlight_datasets/LoL/eval15
- datasets/delowlight_datasets/LoL/our485


## **Denoising**

### **Testing Dataset**
**CBSD68 & Urban100**

In [5]:
noise_base_dir = os.path.join("datasets", "denoising_datasets")

cbsd68_target_dir = os.path.join(noise_base_dir, "CBSD68", "clean")
urban100_target_dir = os.path.join(noise_base_dir, "Urban100_HR", "clean")

os.makedirs(noise_base_dir, exist_ok=True)

def count_images(path):
    if not os.path.exists(path):
        return 0
    count = 0
    for root, _, files in os.walk(path):
        count += len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
    return count

print("Checking CBSD68 + Urban100...")

if count_images(cbsd68_target_dir) > 0 and count_images(urban100_target_dir) > 0:
    print("✓ CBSD68 & Urban100 already exist. Skipping.")
else:
    print("Downloading Super Resolution Benchmarks...")

    cache_path = kagglehub.dataset_download(
        "jesucristo/super-resolution-benchmarks"
    )

    print(f"Cache: {cache_path}")

    cbsd68_source = os.path.join(cache_path, "CBSD68/CBSD68")
    urban100_source = os.path.join(cache_path, "urban100/urban100")

    # ---- CBSD68 ----
    if os.path.exists(cbsd68_source):

        os.makedirs(cbsd68_target_dir, exist_ok=True)

        for f in os.listdir(cbsd68_source):
            src = os.path.join(cbsd68_source, f)
            dst = os.path.join(cbsd68_target_dir, f)

            if os.path.isfile(src):
                shutil.copy2(src, dst)

        print("✓ CBSD68 done")

    # ---- Urban100 ----
    if os.path.exists(urban100_source):

        os.makedirs(urban100_target_dir, exist_ok=True)

        for f in os.listdir(urban100_source):
            src = os.path.join(urban100_source, f)
            dst = os.path.join(urban100_target_dir, f)

            if os.path.isfile(src):
                shutil.copy2(src, dst)

        print("✓ Urban100 done")

Checking CBSD68 + Urban100...


100%|██████████| 923M/923M [01:27<00:00, 11.1MB/s] 

Extracting files...


Cache: /home/mmuuser/.cache/kagglehub/datasets/jesucristo/super-resolution-benchmarks/versions/8
✓ CBSD68 done
✓ Urban100 done


### **Training Dataset**
**BSD400 & WED**

In [6]:
bsd400_target_dir = os.path.join(noise_base_dir, "BSD400", "clean")
wed_target_dir = os.path.join(noise_base_dir, "WED", "clean")

os.makedirs(noise_base_dir, exist_ok=True)

print("\nChecking WED dataset...")

if count_images(wed_target_dir) > 0:
    print("✓ WED already exists. Skipping.")
else:
    print("Downloading WED dataset...")

    wed_cache = kagglehub.dataset_download(
        "hongshenelves/wed-dataset"
    )

    wed_source_dir = os.path.join(wed_cache, "WED")
    if not os.path.exists(wed_source_dir):
        wed_source_dir = wed_cache

    os.makedirs(wed_target_dir, exist_ok=True)

    for item in os.listdir(wed_source_dir):
        src = os.path.join(wed_source_dir, item)
        dst = os.path.join(wed_target_dir, item)

        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)

    print("✓ WED done")

# ===================================
# 3. BSD400 dataset (GitHub)
# ===================================

print("\nChecking BSD400 dataset...")

if count_images(bsd400_target_dir) > 0:
    print("✓ BSD400 already exists. Skipping.")
else:
    print("Downloading BSD400 dataset...")

    github_zip_url = "https://github.com/smartboy110/denoising-datasets/archive/832ae7be88063422bf1b63406e3028c5eae83ac7.zip"

    zip_name = "bsd400.zip"
    extract_dir = "bsd400_temp"

    if os.path.exists(zip_name):
        os.remove(zip_name)
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)

    os.system(f"wget {github_zip_url} -O {zip_name}")
    shutil.unpack_archive(zip_name, extract_dir)

    bsd_source = os.path.join(
        extract_dir,
        "denoising-datasets-832ae7be88063422bf1b63406e3028c5eae83ac7",
        "BSD400"
    )

    if os.path.exists(bsd_source):

        os.makedirs(bsd400_target_dir, exist_ok=True)

        count = 0

        for f in os.listdir(bsd_source):

            if f.lower().endswith(".png"):

                src = os.path.join(bsd_source, f)
                dst = os.path.join(bsd400_target_dir, f)

                shutil.copy2(src, dst)
                count += 1

        print(f"✓ BSD400 done ({count} images)")

    else:
        print("✗ BSD400 not found in archive")

    # cleanup
    if os.path.exists(zip_name):
        os.remove(zip_name)
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)


Checking WED dataset...


100%|██████████| 2.12G/2.12G [03:23<00:00, 11.2MB/s]

Extracting files...


✓ WED done

Checking BSD400 dataset...


--2026-06-23 20:50:40--  https://github.com/smartboy110/denoising-datasets/archive/832ae7be88063422bf1b63406e3028c5eae83ac7.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/smartboy110/denoising-datasets/zip/832ae7be88063422bf1b63406e3028c5eae83ac7 [following]
--2026-06-23 20:50:40--  https://codeload.github.com/smartboy110/denoising-datasets/zip/832ae7be88063422bf1b63406e3028c5eae83ac7
Resolving codeload.github.com (codeload.github.com)... 20.205.243.165
Connecting to codeload.github.com (codeload.github.com)|20.205.243.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘bsd400.zip’

     0K .......... .......... .......... .......... .......... 2.66M
    50K .......... .......... .......... .......... .......... 6.09M
   100K .......... .......... .....

✓ BSD400 done (400 images)


## **Deraining**
Dataset link is in this [github](https://github.com/csdwren/RecDerain/blob/master/datasets/readme.txt) in the [baidu link](https://pan.baidu.com/s/1J0q6Mrno9aMCsaWZUtmbkg#list/path=%2F). If unable to access baidu drive, can get the dataset [here](https://www.swisstransfer.com/d/2cefb9b3-b530-44fb-9f06-81f596a8db85).

In [8]:
rain_base_dir = os.path.join("datasets", "deraining_datasets")

rain_train_dir = os.path.join(rain_base_dir, "RainTrainL")
rain_test_dir = os.path.join(rain_base_dir, "Rain100L")

os.makedirs(rain_base_dir, exist_ok=True)

def count_images(path):
    if not os.path.exists(path):
        return 0
    count = 0
    for root, _, files in os.walk(path):
        count += len([
            f for f in files
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
    return count

if count_images(rain_train_dir) > 0:
    print("✓ RainTrainL already exists. Skipping download.")
else:
    print("RainTrainL not found. Please download manually or add dataset source.")

    print("Sources:")
    print("- Baidu: https://pan.baidu.com/s/1J0q6Mrno9aMCsaWZUtmbkg")
    print("- Alternative: https://www.swisstransfer.com/d/2cefb9b3-b530-44fb-9f06-81f596a8db85")

    os.makedirs(rain_train_dir, exist_ok=True)
    print(f"Place training dataset into: {rain_train_dir}")

if count_images(rain_test_dir) > 0:
    print("✓ Rain100L already exists. Skipping download.")
else:
    print("Rain100L not found. Please download manually or extract dataset.")

    print("Sources:")
    print("- Baidu: https://pan.baidu.com/s/1J0q6Mrno9aMCsaWZUtmbkg")
    print("- Alternative: https://www.swisstransfer.com/d/2cefb9b3-b530-44fb-9f06-81f596a8db85")

    os.makedirs(rain_test_dir, exist_ok=True)
    print(f"Place testing dataset into: {rain_test_dir}")

✓ RainTrainL already exists. Skipping download.
✓ Rain100L already exists. Skipping download.


## **Dataset Validation**

In [24]:
def count_images(path):
    if not os.path.exists(path):
        return None

    count = 0
    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                count += 1
    return count

paths_to_check = {
    # Deblurring
    "Deblur Test Blur": os.path.join(blur_target_dir, "test", "blur"),
    "Deblur Test Sharp": os.path.join(blur_target_dir, "test", "sharp"),
    "Deblur Train Blur": os.path.join(blur_target_dir, "train", "blur"),
    "Deblur Train Sharp": os.path.join(blur_target_dir, "train", "sharp"),

    # Dehazing
    "Dehaze OTS Clear": os.path.join(haze_base_target_dir, "OTS_outdoors", "clear"),
    "Dehaze OTS Hazy": os.path.join(haze_base_target_dir, "OTS_outdoors", "hazy"),
    "Dehaze SOTS GT": os.path.join(haze_base_target_dir, "SOTS_outdoors", "clear"),
    "Dehaze SOTS Hazy": os.path.join(haze_base_target_dir, "SOTS_outdoors", "hazy"),

    # Derain
    "Derain Rain100L Clear": os.path.join(rain_base_dir, "Rain100L", "gt"),
    "Derain Rain100L Rainy": os.path.join(rain_base_dir, "Rain100L", "rainy"),
    "Derain RainTrainL Clear": os.path.join(rain_base_dir, "RainTrainL", "gt"),
    "Derain RainTrainL Rainy": os.path.join(rain_base_dir, "RainTrainL", "rainy"),

    # Low-Light (LoL)
    "Low-Light Eval15 High": os.path.join(lowlight_base_target_dir, "eval15", "high"),
    "Low-Light Eval15 Low": os.path.join(lowlight_base_target_dir, "eval15", "low"),
    "Low-Light Our485 High": os.path.join(lowlight_base_target_dir, "our485", "high"),
    "Low-Light Our485 Low": os.path.join(lowlight_base_target_dir, "our485", "low"),

    # Denoising
    "Denoise BSD400 Clean": bsd400_target_dir,
    "Denoise Urban100 Clean": urban100_target_dir,
    "Denoise CBSD68 Clean": cbsd68_target_dir,
    "Denoise WED Clean": wed_target_dir
}

print(f"{'Dataset Folder':<35} | {'File Count':<10}")
print("-" * 55)

total_files = 0

for name, path in paths_to_check.items():

    if os.path.exists(path):
        num_files = count_images(path)

        if num_files is None:
            print(f"{name:<35} | ERROR ❌")
        else:
            total_files += num_files
            print(f"{name:<35} | {num_files:<10}")

    else:
        print(f"{name:<35} | PATH NOT FOUND ❌")


print("-" * 55)
print(f"{'TOTAL IMAGES':<35} | {total_files:<10}")

Dataset Folder                      | File Count
-------------------------------------------------------
Deblur Test Blur                    | 1111      
Deblur Test Sharp                   | 1111      
Deblur Train Blur                   | 2103      
Deblur Train Sharp                  | 2103      
Dehaze OTS Clear                    | 2061      
Dehaze OTS Hazy                     | 72135     
Dehaze SOTS GT                      | 492       
Dehaze SOTS Hazy                    | 500       
Derain Rain100L Clear               | 100       
Derain Rain100L Rainy               | 100       
Derain RainTrainL Clear             | 200       
Derain RainTrainL Rainy             | 600       
Low-Light Eval15 High               | 15        
Low-Light Eval15 Low                | 15        
Low-Light Our485 High               | 485       
Low-Light Our485 Low                | 485       
Denoise BSD400 Clean                | 400       
Denoise Urban100 Clean              | 100       
Denoise CBSD6

# **Dataset Testing & Training Paths**

In [25]:
def file_exists(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

## **Denoising**

In [26]:
def is_dataset_ready(base_out, dataset_name, sigmas=[15, 25, 50]):
    """
    Check if noisy folders + txt files already exist
    """
    txt_exists = all([
        os.path.exists(os.path.join(base_out, f"{s}_test_paths.txt"))
        for s in sigmas
    ])

    noisy_exists = True
    for s in sigmas:
        noisy_dir = os.path.join(base_out, f"noisy{s}")
        if not os.path.exists(noisy_dir) or len(os.listdir(noisy_dir)) == 0:
            noisy_exists = False
            break

    rand_exists = os.path.exists(os.path.join(base_out, "rand_test_paths.txt"))

    return txt_exists and noisy_exists and rand_exists

In [27]:
def add_gaussian_noise(img, sigma):
    img = np.array(img).astype(np.float32) / 255.0
    noise = np.random.normal(0, sigma / 255.0, img.shape)
    noisy = np.clip(img + noise, 0, 1)
    return (noisy * 255).astype(np.uint8)

In [28]:
def generate_test_dataset(clean_dir, base_out, dataset_name):

    sigmas = [15, 25, 50]

    if is_dataset_ready(base_out, dataset_name):
        print(f"✓ {dataset_name} already exists. Skipping test generation.")
        return

    images = sorted(os.listdir(clean_dir))
    clean_folder_name = "clean"

    for sigma in sigmas:

        noisy_dir = os.path.join(base_out, f"noisy{sigma}")
        os.makedirs(noisy_dir, exist_ok=True)

        txt_path = os.path.join(base_out, f"{sigma}_test_paths.txt")

        with open(txt_path, "w") as f:

            for img_name in images:

                if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    continue

                clean_path = os.path.join(clean_dir, img_name)
                img = Image.open(clean_path).convert("RGB")

                noisy = add_gaussian_noise(img, sigma)

                noisy_path = os.path.join(noisy_dir, img_name)
                Image.fromarray(noisy).save(noisy_path)

                rel_noisy = f"./denoising_datasets/{dataset_name}/noisy{sigma}/{img_name}"
                rel_clean = f"./denoising_datasets/{dataset_name}/{clean_folder_name}/{img_name}"

                f.write(f"{rel_noisy},{rel_clean}\n")

    # random sigma
    rand_txt = os.path.join(base_out, "rand_test_paths.txt")
    rand_dir = os.path.join(base_out, "noisy_rand")
    os.makedirs(rand_dir, exist_ok=True)

    with open(rand_txt, "w") as f:

        for img_name in images:

            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                continue

            sigma = np.random.choice(sigmas)

            clean_path = os.path.join(clean_dir, img_name)
            img = Image.open(clean_path).convert("RGB")

            noisy = add_gaussian_noise(img, sigma)

            noisy_path = os.path.join(rand_dir, img_name)
            Image.fromarray(noisy).save(noisy_path)

            rel_noisy = f"./denoising_datasets/{dataset_name}/noisy_rand/{img_name}"
            rel_clean = f"./denoising_datasets/{dataset_name}/{clean_folder_name}/{img_name}"

            f.write(f"{rel_noisy},{rel_clean}\n")

    print(f"✓ Test dataset ready: {dataset_name}")

In [29]:
def generate_train_dataset(dataset_names, base_out):

    sigmas = [15, 25, 50]

    txt_check = all([
        os.path.exists(os.path.join(base_out, f"{s}_train_paths.txt"))
        for s in sigmas
    ])

    if txt_check:
        print("✓ Train dataset already exists. Skipping generation.")
        return

    txt_files = {}

    for sigma in sigmas:
        txt_path = os.path.join(base_out, f"{sigma}_train_paths.txt")
        txt_files[sigma] = open(txt_path, "w")

    for dataset_name in dataset_names:

        possible_dirs = [
            os.path.join(base_out, dataset_name, "clean"),
            os.path.join(base_out, dataset_name, "original_png")
        ]

        clean_dir = None
        for d in possible_dirs:
            if os.path.exists(d):
                clean_dir = d
                break

        if clean_dir is None:
            print(f"⚠️ Missing dataset: {dataset_name}")
            continue

        images = sorted(os.listdir(clean_dir))
        clean_folder = os.path.basename(os.path.normpath(clean_dir))

        for sigma in sigmas:

            noisy_dir = os.path.join(base_out, dataset_name, f"noisy{sigma}")
            os.makedirs(noisy_dir, exist_ok=True)

            for img_name in images:

                if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    continue

                clean_path = os.path.join(clean_dir, img_name)
                img = Image.open(clean_path).convert("RGB")

                noisy = add_gaussian_noise(img, sigma)

                noisy_path = os.path.join(noisy_dir, img_name)
                Image.fromarray(noisy).save(noisy_path)

                rel_noisy = f"./denoising_datasets/{dataset_name}/noisy{sigma}/{img_name}"
                rel_clean = f"./denoising_datasets/{dataset_name}/{clean_folder}/{img_name}"

                txt_files[sigma].write(f"{rel_noisy},{rel_clean}\n")

    for sigma in sigmas:
        txt_files[sigma].close()

    print("✓ Train dataset ready (all datasets)")

In [30]:
generate_test_dataset(
    cbsd68_target_dir,
    os.path.join(noise_base_dir, "CBSD68"),
    "CBSD68"
)

generate_test_dataset(
    urban100_target_dir,
    os.path.join(noise_base_dir, "Urban100_HR"),
    "Urban100_HR"
)

generate_train_dataset(["BSD400", "WED"], noise_base_dir)

✓ Test dataset ready: CBSD68
✓ Test dataset ready: Urban100_HR
✓ Train dataset ready (all datasets)


## **Delowlight**

In [31]:
# train paths
train_txt = os.path.join(lowlight_base_target_dir, "train_paths.txt")

if file_exists(train_txt):
    print("✓ LoL train_paths.txt already exists. Skipping.")
else:
    print("Generating LoL train_paths.txt...")

    low_dir = os.path.join(lowlight_base_target_dir, "our485", "low")
    high_dir = os.path.join(lowlight_base_target_dir, "our485", "high")

    relative_low = "./delowlight_datasets/LoL/our485/low"
    relative_high = "./delowlight_datasets/LoL/our485/high"

    files = sorted(os.listdir(low_dir))

    with open(train_txt, "w") as f:
        for name in files:

            low_path = os.path.join(relative_low, name)
            high_path = os.path.join(relative_high, name)

            f.write(f"{low_path},{high_path}\n")

    print("✓ Train paths generated")


# test paths
test_txt = os.path.join(lowlight_base_target_dir, "test_paths.txt")

if file_exists(test_txt):
    print("✓ LoL test_paths.txt already exists. Skipping.")
else:
    print("Generating LoL test_paths.txt...")

    low_dir = os.path.join(lowlight_base_target_dir, "eval15", "low")
    high_dir = os.path.join(lowlight_base_target_dir, "eval15", "high")

    relative_low = "./delowlight_datasets/LoL/eval15/low"
    relative_high = "./delowlight_datasets/LoL/eval15/high"

    files = sorted(os.listdir(low_dir))

    with open(test_txt, "w") as f:
        for name in files:

            low_path = os.path.join(relative_low, name)
            high_path = os.path.join(relative_high, name)

            f.write(f"{low_path},{high_path}\n")

    print("✓ Test paths generated")

Generating LoL train_paths.txt...
✓ Train paths generated
Generating LoL test_paths.txt...
✓ Test paths generated


## **Deblurring**

In [32]:
# train paths
train_txt = os.path.join(blur_target_dir, "train_paths.txt")

if file_exists(train_txt):
    print("✓ GoPro train_paths.txt already exists. Skipping.")
else:
    print("Generating GoPro train_paths.txt...")

    low_dir = os.path.join(blur_target_dir, "train", "blur")
    high_dir = os.path.join(blur_target_dir, "train", "sharp")

    relative_low = "./deblurring_datasets/GoPro/train/blur"
    relative_high = "./deblurring_datasets/GoPro/train/sharp"

    files = sorted(os.listdir(low_dir))

    with open(train_txt, "w") as f:
        for name in files:

            low_path = os.path.join(relative_low, name)
            high_path = os.path.join(relative_high, name)

            f.write(f"{low_path},{high_path}\n")

    print("✓ Train paths generated")


# test paths
test_txt = os.path.join(blur_target_dir, "test_paths.txt")

if file_exists(test_txt):
    print("✓ GoPro test_paths.txt already exists. Skipping.")
else:
    print("Generating GoPro test_paths.txt...")

    low_dir = os.path.join(blur_target_dir, "test", "blur")
    high_dir = os.path.join(blur_target_dir, "test", "sharp")

    relative_low = "./deblurring_datasets/GoPro/test/blur"
    relative_high = "./deblurring_datasets/GoPro/test/sharp"

    files = sorted(os.listdir(low_dir))

    with open(test_txt, "w") as f:
        for name in files:

            low_path = os.path.join(relative_low, name)
            high_path = os.path.join(relative_high, name)

            f.write(f"{low_path},{high_path}\n")

    print("✓ Test paths generated")

Generating GoPro train_paths.txt...
✓ Train paths generated
Generating GoPro test_paths.txt...
✓ Test paths generated


## **Dehazing**

In [33]:
# train paths
train_txt = os.path.join(haze_base_target_dir, "train_paths.txt")

if file_exists(train_txt):
    print("✓ OTS train_paths.txt already exists. Skipping.")
else:
    print("Generating OTS train_paths.txt...")

    low_dir = os.path.join(haze_base_target_dir, "OTS_outdoors", "hazy")
    high_dir = os.path.join(haze_base_target_dir, "OTS_outdoors", "clear")

    relative_low = "./dehazing_datasets/OTS_outdoors/hazy"
    relative_high = "./dehazing_datasets/OTS_outdoors/clear"

    clear_files = os.listdir(high_dir)
    clear_map = {os.path.splitext(f)[0]: f for f in clear_files}

    hazy_files = sorted(os.listdir(low_dir))

    with open(train_txt, "w") as f:

        for name in hazy_files:

            base_id = name.split('_')[0]

            if base_id in clear_map:

                clear_name = clear_map[base_id]

                low_path = os.path.join(relative_low, name)
                high_path = os.path.join(relative_high, clear_name)

                f.write(f"{low_path},{high_path}\n")

    print("✓ OTS train paths generated")


# test paths
test_txt = os.path.join(haze_base_target_dir, "test_paths.txt")

if file_exists(test_txt):
    print("✓ SOTS test_paths.txt already exists. Skipping.")
else:
    print("Generating SOTS test_paths.txt...")

    low_dir = os.path.join(haze_base_target_dir, "SOTS_outdoors", "hazy")
    high_dir = os.path.join(haze_base_target_dir, "SOTS_outdoors", "clear")

    relative_low = "./dehazing_datasets/SOTS_outdoors/hazy"
    relative_high = "./dehazing_datasets/SOTS_outdoors/clear"

    clear_files_test = os.listdir(high_dir)
    clear_map_test = {os.path.splitext(f)[0]: f for f in clear_files_test}

    hazy_files_test = sorted(os.listdir(low_dir))

    with open(test_txt, "w") as f:

        for name in hazy_files_test:

            base_id = name.split('_')[0]

            if base_id in clear_map_test:

                clear_name = clear_map_test[base_id]

                low_path = os.path.join(relative_low, name)
                high_path = os.path.join(relative_high, clear_name)

                f.write(f"{low_path},{high_path}\n")

    print("✓ SOTS test paths generated")

Generating OTS train_paths.txt...
✓ OTS train paths generated
Generating SOTS test_paths.txt...
✓ SOTS test paths generated


## **Deraining**

In [34]:
# train paths
train_txt = os.path.join(rain_base_dir, "Rain100L", "train_paths.txt")

if file_exists(train_txt):
    print("✓ RainTrainL train_paths.txt already exists. Skipping.")
else:
    print("Generating RainTrainL train paths...")

    train_low_dir = os.path.join(rain_base_dir, "RainTrainL", "rainy")
    train_high_dir = os.path.join(rain_base_dir, "RainTrainL", "gt")

    train_relative_low = "./deraining_datasets/RainTrainL/rainy"
    train_relative_high = "./deraining_datasets/RainTrainL/gt"

    train_pairs_count = 0

    if os.path.exists(train_low_dir) and os.path.exists(train_high_dir):

        with open(train_txt, "w") as f:

            for name in sorted(os.listdir(train_low_dir)):

                if not name.lower().endswith(".png"):
                    continue

                try:
                    img_num = name.split('-')[1].split('.')[0]
                    gt_name = f"norain-{img_num}.png"

                    gt_path = os.path.join(train_high_dir, gt_name)

                    if os.path.exists(gt_path):

                        low_path = os.path.join(train_relative_low, name)
                        high_path = os.path.join(train_relative_high, gt_name)

                        f.write(f"{low_path},{high_path}\n")
                        train_pairs_count += 1

                except IndexError:
                    continue

        print(f"✓ RainTrainL done ({train_pairs_count} pairs)")

    else:
        print("❌ RainTrainL folders not found")


# test paths
test_txt = os.path.join(rain_base_dir, "Rain100L", "test_paths.txt")

if file_exists(test_txt):
    print("✓ Rain100L test_paths.txt already exists. Skipping.")
else:
    print("Generating Rain100L test paths...")

    test_low_dir = os.path.join(rain_base_dir, "Rain100L", "rainy")
    test_high_dir = os.path.join(rain_base_dir, "Rain100L", "gt")

    test_relative_low = "./deraining_datasets/Rain100L/rainy"
    test_relative_high = "./deraining_datasets/Rain100L/gt"

    test_pairs_count = 0

    if os.path.exists(test_low_dir) and os.path.exists(test_high_dir):

        with open(test_txt, "w") as f:

            for name in sorted(os.listdir(test_low_dir)):

                if not name.lower().endswith(".png"):
                    continue

                try:
                    img_num = name.split('-')[1].split('.')[0]
                    gt_name = f"norain-{img_num}.png"

                    gt_path = os.path.join(test_high_dir, gt_name)

                    if os.path.exists(gt_path):

                        low_path = os.path.join(test_relative_low, name)
                        high_path = os.path.join(test_relative_high, gt_name)

                        f.write(f"{low_path},{high_path}\n")
                        test_pairs_count += 1

                except IndexError:
                    continue

        print(f"✓ Rain100L done ({test_pairs_count} pairs)")

    else:
        print("❌ Rain100L folders not found")

Generating RainTrainL train paths...
✓ RainTrainL done (600 pairs)
Generating Rain100L test paths...
✓ Rain100L done (100 pairs)


## **Dataset Train & Test Paths Validation**

In [35]:
paths_to_check = {
    # --- Denoising Train ---
    "Denoising Train (Sigma 15)": os.path.join(noise_base_dir, "15_train_paths.txt"),
    "Denoising Train (Sigma 25)": os.path.join(noise_base_dir, "25_train_paths.txt"),
    "Denoising Train (Sigma 50)": os.path.join(noise_base_dir, "50_train_paths.txt"),

    # --- Denoising Test (CBSD68) ---
    "Denoising Test (CBSD68 S15)": os.path.join(noise_base_dir, "CBSD68", "15_test_paths.txt"),
    "Denoising Test (CBSD68 S25)": os.path.join(noise_base_dir, "CBSD68", "25_test_paths.txt"),
    "Denoising Test (CBSD68 S50)": os.path.join(noise_base_dir, "CBSD68", "50_test_paths.txt"),
    "Denoising Test (CBSD68 Rand)": os.path.join(noise_base_dir, "CBSD68", "rand_test_paths.txt"),

    # --- Denoising Test (Urban100_HR) ---
    "Denoising Test (Urban100 S15)": os.path.join(noise_base_dir, "Urban100_HR", "15_test_paths.txt"),
    "Denoising Test (Urban100 S25)": os.path.join(noise_base_dir, "Urban100_HR", "25_test_paths.txt"),
    "Denoising Test (Urban100 S50)": os.path.join(noise_base_dir, "Urban100_HR", "50_test_paths.txt"),
    "Denoising Test (Urban100 Rand)": os.path.join(noise_base_dir, "Urban100_HR", "rand_test_paths.txt"),

    # --- Lowlight (LoL) ---
    "Lowlight Train (LoL)": os.path.join(lowlight_base_target_dir, "train_paths.txt"),
    "Lowlight Test (LoL)":  os.path.join(lowlight_base_target_dir, "test_paths.txt"),

    # --- Deblurring (GoPro) ---
    "Deblurring Train (GoPro)": os.path.join(blur_target_dir, "train_paths.txt"),
    "Deblurring Test (GoPro)":  os.path.join(blur_target_dir, "test_paths.txt"),

    # --- Dehazing (OTS / SOTS) ---
    "Dehazing Train (OTS)": os.path.join(haze_base_target_dir, "train_paths.txt"),
    "Dehazing Test (SOTS)": os.path.join(haze_base_target_dir, "test_paths.txt"),

    # --- Deraining (Rain100L / RainTrainL) ---
    "Deraining Train (RainTrainL)": os.path.join(rain_base_dir, "Rain100L", "train_paths.txt"),
    "Deraining Test (Rain100L)": os.path.join(rain_base_dir, "Rain100L", "test_paths.txt")
}

print("=== Checking Dataset Text Path Lengths ===\n")

for dataset_name, file_path in paths_to_check.items():
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            # Filter out empty or whitespace-only lines
            lines = [line for line in f.read().splitlines() if line.strip()]
            print(f"✅ {dataset_name:<30}: {len(lines):,} pairs")
    else:
        print(f"❌ {dataset_name:<30}: File not found at target path.")

=== Checking Dataset Text Path Lengths ===

✅ Denoising Train (Sigma 15)    : 5,143 pairs
✅ Denoising Train (Sigma 25)    : 5,143 pairs
✅ Denoising Train (Sigma 50)    : 5,143 pairs
✅ Denoising Test (CBSD68 S15)   : 68 pairs
✅ Denoising Test (CBSD68 S25)   : 68 pairs
✅ Denoising Test (CBSD68 S50)   : 68 pairs
✅ Denoising Test (CBSD68 Rand)  : 68 pairs
✅ Denoising Test (Urban100 S15) : 100 pairs
✅ Denoising Test (Urban100 S25) : 100 pairs
✅ Denoising Test (Urban100 S50) : 100 pairs
✅ Denoising Test (Urban100 Rand): 100 pairs
✅ Lowlight Train (LoL)          : 485 pairs
✅ Lowlight Test (LoL)           : 15 pairs
✅ Deblurring Train (GoPro)      : 2,103 pairs
✅ Deblurring Test (GoPro)       : 1,111 pairs
✅ Dehazing Train (OTS)          : 72,135 pairs
✅ Dehazing Test (SOTS)          : 500 pairs
✅ Deraining Train (RainTrainL)  : 600 pairs
✅ Deraining Test (Rain100L)     : 100 pairs
